Group Members:

Student 1 Name : Irdina Nadiah Binti Mohd Rizal, ID : IS01083874

Student 2 Name : Leena Dewi Thevan, ID : IS01083877

Student 3 Name : Zalin Zalifah Binti Zulkifli, ID : IS01083447

In [16]:
import nltk 
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer 
from nltk.stem import PorterStemmer

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import pandas as pd 
 
nltk.download('stopwords') 
nltk.download('punkt') 
nltk.download('wordnet') 

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/e71362c6-faaf-4294-bd6d-
[nltk_data]     c1e0fe8cb3c7/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/e71362c6-faaf-4294-bd6d-
[nltk_data]     c1e0fe8cb3c7/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/e71362c6-faaf-4294-bd6d-
[nltk_data]     c1e0fe8cb3c7/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [17]:
df = pd.read_csv('news_dataset.csv') 

df = df[['text']].dropna()

documents = df['text'].tolist()

In [20]:
stop_words = set(stopwords.words('english'))  
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer() 
 
def preprocess_text(text): 
    tokens = word_tokenize(text.lower())  
    tokens = [token for token in tokens if token.isalnum()] 
    tokens = [token for token in tokens if token not in stop_words] 
    tokens = [stemmer.stem(token) for token in tokens]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]  
    return tokens  
 
preprocessed_documents = [preprocess_text(doc) for doc in documents] 
 
print(preprocessed_documents[0])

['wonder', 'anyon', 'could', 'enlighten', 'car', 'saw', 'day', 'sport', 'car', 'look', 'late', 'earli', '70', 'call', 'bricklin', 'door', 'realli', 'small', 'addit', 'front', 'bumper', 'separ', 'rest', 'bodi', 'know', 'anyon', 'tellm', 'model', 'name', 'engin', 'spec', 'year', 'product', 'car', 'made', 'histori', 'whatev', 'info', 'funki', 'look', 'car', 'plea']


In [21]:
dictionary = corpora.Dictionary(preprocessed_documents)  
 
dictionary.filter_extremes(no_below=15, no_above=0.5) 
 
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]   

In [22]:
lda_model = LdaModel(corpus, num_topics=4, id2word=dictionary, passes=15)  

In [12]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=preprocessed_documents,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print("Coherence Score:", coherence_score)


Coherence Score: 0.6453991257461758


In [13]:
article_labels = [] 
 
for i, doc in enumerate(preprocessed_documents): 
    bow = dictionary.doc2bow(doc) 
    topics = lda_model.get_document_topics(bow) 
    dominant_topic = max(topics, key=lambda x: x[1])[0] 
    article_labels.append(dominant_topic) 
 
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels}) 
 
print("Table with Articles and Topic:") 
print(df_result) 
print()

Table with Articles and Topic:
                                                 Article  Topic
0      I was wondering if anyone out there could enli...      3
1      I recently posted an article asking what kind ...      3
2      \nIt depends on your priorities.  A lot of peo...      3
3      an excellent automatic can be found in the sub...      3
4      : Ford and his automobile.  I need information...      3
...                                                  ...    ...
11091  Secrecy in Clipper Chip\n\nThe serial number o...      0
11092  Hi !\n\nI am interested in the source of FEAL ...      0
11093  The actual algorithm is classified, however, t...      0
11094  \n\tThis appears to be generic calling upon th...      1
11095  \nProbably keep quiet and take it, lest they g...      3

[11096 rows x 2 columns]



In [15]:
for topic_id in range(lda_model.num_topics): 
    print(f"Top terms for Topic #{topic_id}:") 
    top_terms = lda_model.show_topic(topic_id, topn=10) 
    print([term[0] for term in top_terms]) 
    print()


Top terms for Topic #0:
['use', 'key', 'file', 'system', 'encrypt', 'program', 'chip', 'one', 'get', 'inform']

Top terms for Topic #1:
['peopl', 'would', 'one', 'say', 'govern', 'think', 'god', 'know', 'right', 'law']

Top terms for Topic #2:
['1', 'x', 'q', '0', 'max', '2', '7', 'g', 'r', '3']

Top terms for Topic #3:
['year', 'go', 'would', 'get', 'one', 'like', 'time', 'game', 'think', 'know']



Result Interpretation

A coherence score of 0.645 indicates that the LDA model generated topics that are meaningful and more interpretable. In topic modeling, coherence scores range from 0 to 1. The higher the coherence score, the better the quality and clarity of the topics. Therefore, a coherence score of 0.645 suggests that the words grouped under each topic are reasonably related, making the topics useful for analysis. However, there is still room for improvement as a higher coherence value would indicate even clearer and more interpretable distinct topics.